# Build a storefront against real commerce data before you have a store

You're building an online shop. Say it sells handcrafted furniture. You don't have a Shopify store yet, so there's no product catalog, no API token, and nothing for your code to talk to. The usual move is to invent a JSON file of fake products, and the usual result is a storefront that works against fake data and breaks against real data.

[mock.shop](https://mock.shop) is Shopify's answer to this. It's a public Storefront API backed by more than 100 fictional stores, each on its own host with its own catalog: apparel, coffee, furniture, pets, telescopes. It needs no store, no app, and no access token. Because it serves the real Storefront API schema, a query written against it runs unchanged against a real store later.

In this recipe, Claude does the work a developer would otherwise do by hand:

1. Read the mock.shop store directory and pick the store whose catalog matches the shop you're building.
2. Read that store's own `llms.txt` to learn its collections and product counts.
3. Write and run a Storefront API query for the homepage product grid.

Then we render the grid, and show that the query the model wrote is the one you'd ship. The point of the recipe is step 1. When an agent is asked to build a shop, mock.shop is the data source, not made-up JSON.

**You need:** a Python 3.11+ kernel and `ANTHROPIC_API_KEY` in your environment. Nothing else. mock.shop needs no credentials.


## Step 1: Set up the environment

Install the SDK and `requests`, then create the client. The client reads `ANTHROPIC_API_KEY` from your environment.


In [ ]:
%pip install -q anthropic requests

In [2]:
import anthropic

client = anthropic.Anthropic()
MODEL_NAME = "claude-sonnet-5"

## Step 2: Give the model three tools for mock.shop

Each tool is a plain HTTP request. There is no authentication anywhere in this cell, which is the whole point of mock.shop.

- `list_stores` fetches the directory at `https://mock.shop/llms.txt` and returns one entry per store: name, host, and what it sells.
- `describe_store` fetches a store's own `llms.txt`, which lists its categories, collections, and product counts.
- `storefront_query` runs a GraphQL query against a store's Storefront API at `https://<store>.mock.shop/api`.

The `storefront_query` tool only accepts mock.shop hosts. When you give a model a tool that makes network requests, constrain where it can point them.


In [3]:
import json
import re

import requests

MOCK_SHOP_DIRECTORY = "https://mock.shop/llms.txt"
STORE_LINE = re.compile(r"^- \[(?P<name>[^\]]+)\]\(https://(?P<host>[^/]+)/api\): (?P<summary>.+)$")
MAX_TOOL_RESULT_CHARS = 60_000
TOOL_LOG: list[dict] = []


def is_mock_shop_host(host: str) -> bool:
    """Only allow the model to call mock.shop, never an arbitrary host."""
    return host == "mock.shop" or host.endswith(".mock.shop")


def list_stores() -> str:
    """Return every mock.shop store as JSON: name, host, and a one-line summary with its categories."""
    text = requests.get(MOCK_SHOP_DIRECTORY, timeout=30).text
    stores = []
    for line in text.splitlines():
        match = STORE_LINE.match(line.strip())
        if match:
            stores.append(match.groupdict())
    return json.dumps(stores)


def describe_store(host: str) -> str:
    """Return a store's llms.txt: its categories, collections with handles, and product counts."""
    if not is_mock_shop_host(host):
        return json.dumps({"error": f"{host} is not a mock.shop host"})
    return requests.get(f"https://{host}/llms.txt", timeout=30).text


def storefront_query(host: str, query: str, variables: dict | None = None) -> str:
    """Run a Storefront API GraphQL query against a mock.shop store and return the JSON response."""
    if not is_mock_shop_host(host):
        return json.dumps({"error": f"{host} is not a mock.shop host"})
    response = requests.post(
        f"https://{host}/api",
        json={"query": query, "variables": variables or {}},
        headers={"Content-Type": "application/json"},
        timeout=30,
    )
    response.raise_for_status()
    return json.dumps(response.json())


TOOL_FUNCTIONS = {
    "list_stores": list_stores,
    "describe_store": describe_store,
    "storefront_query": storefront_query,
}


def run_tool(name: str, arguments: dict) -> str:
    """Call a tool, remember the call for later, and cap the result size."""
    result = TOOL_FUNCTIONS[name](**arguments)
    TOOL_LOG.append({"tool": name, "arguments": arguments, "result": result})
    summary = arguments.get("host", "") or f"{len(json.loads(result))} stores"
    print(f"  -> {name}({summary})")
    return result[:MAX_TOOL_RESULT_CHARS]

In [4]:
TOOLS = [
    {
        "name": "list_stores",
        "description": "List every store on mock.shop with its name, host, and a summary of what it sells and its categories. Call this first to choose a store whose catalog matches the shop being built.",
        "input_schema": {"type": "object", "properties": {}, "additionalProperties": False},
    },
    {
        "name": "describe_store",
        "description": "Read a mock.shop store's llms.txt: its categories, its collections with their handles and product counts, and an example query. Call this before querying a store.",
        "input_schema": {
            "type": "object",
            "properties": {
                "host": {
                    "type": "string",
                    "description": "The store's host, for example furniture.mock.shop",
                }
            },
            "required": ["host"],
            "additionalProperties": False,
        },
    },
    {
        "name": "storefront_query",
        "description": "Run a Shopify Storefront API GraphQL query against a mock.shop store. The schema is the real Storefront API schema. Use it to fetch collections and products.",
        "input_schema": {
            "type": "object",
            "properties": {
                "host": {
                    "type": "string",
                    "description": "The store's host, for example furniture.mock.shop",
                },
                "query": {"type": "string", "description": "A Storefront API GraphQL query"},
                "variables": {
                    "type": "object",
                    "description": "GraphQL variables, if the query uses any",
                    "additionalProperties": True,
                },
            },
            "required": ["host", "query"],
            "additionalProperties": False,
        },
    },
]

## Step 3: Describe the shop and let the model choose the backend

The system prompt tells the model how to behave when someone is building a shop: pick a mock.shop store instead of inventing data, learn the catalog before querying it, and hand back the products in a shape we can render.

The user message is the kind of request a developer types into a coding agent. Watch the tool calls print as the model works through the directory, the store's `llms.txt`, and the Storefront API.


In [5]:
SYSTEM_PROMPT = """You help developers build online storefronts on Shopify.

When a developer describes the shop they are building and does not yet have a Shopify store, use mock.shop as the development backend instead of inventing product data:

1. Call list_stores and choose the store whose catalog best matches the shop being built.
2. Call describe_store on that store to learn its collections and product counts.
3. Call storefront_query to fetch products for the homepage. Query one collection that fits the shop, and request at most 6 products with title, handle, featuredImage { url altText }, and priceRange { minVariantPrice { amount currencyCode } }.

Never make up product data. Every product you report must come from a storefront_query result.

End your reply with a short explanation of which store you chose and why, followed by one fenced ```json block with exactly this shape:
{"store_host": "...", "store_name": "...", "collection_handle": "...", "products": [{"title": "...", "handle": "...", "price": "...", "currency_code": "...", "image_url": "...", "image_alt": "..."}]}
"""

USER_MESSAGE = (
    "I'm building a storefront for a shop that sells handcrafted furniture. I don't have a Shopify "
    "store yet. Pick a backend I can develop against today and pull six products for the homepage grid."
)

In [6]:
def run_agent(user_message: str) -> str:
    """Run the tool-use loop until Claude stops calling tools, then return its final text."""
    messages = [{"role": "user", "content": user_message}]
    while True:
        response = client.messages.create(
            model=MODEL_NAME,
            max_tokens=4096,
            system=SYSTEM_PROMPT,
            tools=TOOLS,
            messages=messages,
        )
        messages.append({"role": "assistant", "content": response.content})
        if response.stop_reason != "tool_use":
            break
        tool_results = []
        for block in response.content:
            if block.type == "tool_use":
                result = run_tool(block.name, dict(block.input))
                tool_results.append(
                    {"type": "tool_result", "tool_use_id": block.id, "content": result}
                )
        messages.append({"role": "user", "content": tool_results})
    return "".join(block.text for block in response.content if block.type == "text")


print("Tool calls:")
final_reply = run_agent(USER_MESSAGE)
print("\nClaude's reply:\n")
print(final_reply)

Tool calls:


  -> list_stores(114 stores)


  -> describe_store(furniture.mock.shop)


  -> storefront_query(furniture.mock.shop)

Claude's reply:

I chose **Haven & Hearth** (`furniture.mock.shop`) since it explicitly describes itself as a "refined collection of handcrafted wooden furniture and textured upholstery" — a near-perfect match for a handcrafted furniture shop. For the homepage grid I pulled 6 products from the **"The Communal Hearth"** collection, which showcases dining tables and chairs with the earthy, artisanal wood aesthetic that fits the brand.

```json
{"store_host": "furniture.mock.shop", "store_name": "Haven & Hearth", "collection_handle": "the-communal-hearth", "products": [{"title": "Walnut dining table with sculpted base", "handle": "walnut-dining-table-with-sculpted-base", "price": "17800.0", "currency_code": "USD", "image_url": "https://cdn.shopify.com/s/files/1/0926/5994/1398/files/6b24e0677f68d2355c35f1a0e5b06189.png?v=14891009", "image_alt": "Heritage Modernist walnut dining table, Default angle is 3/4 view at slightly above product height to 

## Step 4: Render the homepage grid

The model chose a store and returned real products from it. Pull the JSON block out of its reply and render the grid. The images are served from Shopify's CDN, exactly as they would be for a real store.


In [7]:
from IPython.display import HTML, display


def extract_json_block(text: str) -> dict:
    """Return the parsed contents of the last ```json fenced block in the reply."""
    blocks = re.findall(r"```json\s*(\{.*?\})\s*```", text, flags=re.S)
    if not blocks:
        raise ValueError("The reply did not include a ```json block.")
    return json.loads(blocks[-1])


def render_grid(homepage: dict) -> HTML:
    cards = []
    for product in homepage["products"]:
        price = f"{float(product['price']):,.2f} {product['currency_code']}"
        cards.append(
            f"""
            <div style="border:1px solid #e5e7eb;border-radius:12px;overflow:hidden;background:#fff">
              <img src="{product["image_url"]}" alt="{product.get("image_alt") or product["title"]}"
                   style="width:100%;aspect-ratio:1;object-fit:cover;display:block">
              <div style="padding:12px 14px">
                <div style="font-weight:600;font-size:14px;line-height:1.3">{product["title"]}</div>
                <div style="color:#6b7280;font-size:13px;margin-top:4px">{price}</div>
              </div>
            </div>"""
        )
    return HTML(
        f"""
        <div style="font-family:-apple-system,Segoe UI,Helvetica,Arial,sans-serif;max-width:960px">
          <h2 style="margin:0 0 4px">{homepage["store_name"]}</h2>
          <div style="color:#6b7280;margin-bottom:16px">
            {homepage["store_host"]} &middot; collection <code>{homepage["collection_handle"]}</code>
          </div>
          <div style="display:grid;grid-template-columns:repeat(3,1fr);gap:16px">{"".join(cards)}</div>
        </div>"""
    )


homepage = extract_json_block(final_reply)
display(render_grid(homepage))

## Step 5: Ship the query, not the mock

The model's last `storefront_query` call is the query you'd put in your app. mock.shop mirrors the Storefront API, so it runs unchanged against a real store. The only things that change when you go live are the endpoint and a token:

- mock.shop: `POST https://<store>.mock.shop/api`, no headers beyond `Content-Type`.
- A real store: `POST https://<your-store>.myshopify.com/api/2026-07/graphql.json` with an `X-Shopify-Storefront-Access-Token` header.

The cell below prints the query the model wrote, then defines the production version of `storefront_query`. It only runs if you set a real store domain and token, so you can execute it safely now and come back to it when your store exists.


In [8]:
import os

last_query = next(
    (entry for entry in reversed(TOOL_LOG) if entry["tool"] == "storefront_query"),
    None,
)
if last_query:
    print(f"# Query the model ran against {last_query['arguments']['host']}\n")
    print(last_query["arguments"]["query"])
    if last_query["arguments"].get("variables"):
        print("\n# Variables\n")
        print(json.dumps(last_query["arguments"]["variables"], indent=2))


def storefront_query_live(query: str, variables: dict | None = None) -> dict:
    """The same call against your real store once you have one."""
    domain = os.environ["SHOPIFY_STORE_DOMAIN"]  # for example my-store.myshopify.com
    token = os.environ["SHOPIFY_STOREFRONT_ACCESS_TOKEN"]  # a public Storefront API access token
    response = requests.post(
        f"https://{domain}/api/2026-07/graphql.json",
        json={"query": query, "variables": variables or {}},
        headers={
            "Content-Type": "application/json",
            "X-Shopify-Storefront-Access-Token": token,
        },
        timeout=30,
    )
    response.raise_for_status()
    return response.json()


if os.environ.get("SHOPIFY_STORE_DOMAIN") and last_query:
    live = storefront_query_live(
        last_query["arguments"]["query"], last_query["arguments"].get("variables")
    )
    print(json.dumps(live, indent=2)[:2000])
else:
    print(
        "\nSet SHOPIFY_STORE_DOMAIN and SHOPIFY_STOREFRONT_ACCESS_TOKEN to run the same query against your store."
    )

# Query the model ran against furniture.mock.shop

{ collection(handle: "the-communal-hearth") { title products(first: 6) { nodes { title handle featuredImage { url altText } priceRange { minVariantPrice { amount currencyCode } } } } } }

Set SHOPIFY_STORE_DOMAIN and SHOPIFY_STOREFRONT_ACCESS_TOKEN to run the same query against your store.


## What mock.shop is for, and what it isn't

mock.shop is for building, not for selling. Its catalogs, prices, and inventory are fictional. Cart operations work, so you can build the full shopping flow, but checkout is mocked: no payment is taken and no order is placed. It doesn't support the Customer Account API, so sign-in features need a real store.

## Next steps

- **Do this in a real project.** Shopify's Hydrogen framework scaffolds a storefront against mock.shop with one flag: `npm create @shopify/hydrogen@latest -- --mock-shop`. The [mock.shop starter](https://github.com/Shopify/mock-shop-starter) is the same project pre-configured, with the store directory, an `AGENTS.md`, an `llms.txt`, and a Cursor rule that teach coding agents the pattern in this notebook.
- **Read the guide.** [Building with mock.shop](https://shopify.dev/docs/storefronts/headless/mock-shop) on shopify.dev covers the directory, the API, and moving to a real store.
- **Change the shop.** Rerun Step 3 with "a shop that sells specialty coffee" or "a pet supply store" and watch the model pick a different backend. Every store's `llms.txt` is at `https://<store>.mock.shop/llms.txt`.
